# 02_2 PseudoGT Visual Compare

GT付き画像に対して、同じ画像を3列で比較します。

1. 通常のGT
2. モデルの生推論結果を低閾値NMSで載せたもの
3. EfficientTeacher設定でpseudoGTとして残るもの

初期値は30枚です。`NUM_IMAGES`を変えると出力枚数を調整できます。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    markers = (
        "efficient_teacher/pseudogt_visual_compare_02_2.py",
        "navigating_data_heterogeneity/vendor/efficientteacher/train.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in markers):
                return candidate
    raise FileNotFoundError("Object_Detection repository root was not found.")


REPO_ROOT = find_repo_root()
ET_NOTEBOOK_ROOT = REPO_ROOT / "efficient_teacher"
if str(ET_NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(ET_NOTEBOOK_ROOT))

from pseudogt_visual_compare_02_2 import (  # noqa: E402
    PseudoGTVisualComparer,
    VisualCompareConfig,
    default_image_list,
    warmup_checkpoint,
)

pd.options.display.max_columns = 120
print("repo:", REPO_ROOT)

repo: /app/Object_Detection


## 1. 設定

`VARIANT`を変えるとpseudoGT側の閾値・cap設定が変わります。GT比較が目的なので、初期値ではserver validation画像を使います。

In [2]:
EXP_ROOT = REPO_ROOT / "efficient_teacher" / "efficientteacher_pseudogt_dqa_probe_2h"

NUM_IMAGES = 30
VARIANT = "et_default_lr1e-3"
CLIENT_ID = 0

IMAGE_LIST = default_image_list(EXP_ROOT)
WEIGHTS = warmup_checkpoint()
OUTPUT_DIR = EXP_ROOT / "results" / "visual_compare" / VARIANT

START_INDEX = 0
SAMPLE_MODE = "first"  # "first" or "random"
RANDOM_SEED = 0

RAW_CONF_THRES = 0.001
MAX_RAW_DETECTIONS = 300
SHOW_IGNORED_PSEUDO = False

PANEL_WIDTH = 960
PREVIEW_WIDTH = PANEL_WIDTH * 3 + 16
DRAW_RAW_LABELS = False
DRAW_PSEUDO_LABELS = True
DRAW_GT_LABELS = True

DEVICE = ""  # "" lets EfficientTeacher choose; use "cpu" if needed
HALF = False

cfg = VisualCompareConfig(
    experiment_root=EXP_ROOT,
    variant=VARIANT,
    client_id=CLIENT_ID,
    image_list=IMAGE_LIST,
    weights=WEIGHTS,
    output_dir=OUTPUT_DIR,
    num_images=NUM_IMAGES,
    start_index=START_INDEX,
    sample_mode=SAMPLE_MODE,
    random_seed=RANDOM_SEED,
    raw_conf_thres=RAW_CONF_THRES,
    max_raw_detections=MAX_RAW_DETECTIONS,
    show_ignored_pseudo=SHOW_IGNORED_PSEUDO,
    panel_width=PANEL_WIDTH,
    draw_raw_labels=DRAW_RAW_LABELS,
    draw_pseudo_labels=DRAW_PSEUDO_LABELS,
    draw_gt_labels=DRAW_GT_LABELS,
    device=DEVICE,
    half=HALF,
)
comparer = PseudoGTVisualComparer(cfg)
display(comparer.describe())

{'image_list': '/app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/mini_lists/server_cloudy_val_512.txt',
 'weights': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/global_checkpoints/round000_warmup.pt',
 'output_dir': '/app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/visual_compare/et_default_lr1e-3',
 'variant': 'et_default_lr1e-3',
 'client_id': 0,
 'num_images': 30,
 'raw_conf_thres': 0.001,
 'raw_iou_thres': 0.65,
 'pseudo_nms_conf_thres': 0.1,
 'pseudo_iou_thres': 0.65,
 'ignore_thres_low': 0.1,
 'ignore_thres_high': 0.6,
 'max_pseudo_per_image': 0,
 'max_pseudo_per_class_image': 0}

## 2. 画像出力

実行すると横3列の比較画像が`OUTPUT_DIR`に保存され、summary CSVも同じ場所に保存されます。

In [3]:
summary = comparer.run()
display(summary)
print("output_dir:", comparer.output_dir)

EfficientTeacher  48b5cc2 torch 2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
/root/micromamba/envs/al_yolov8/lib/python3.10/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


,index,image,label,output,gt_count,raw_count,pseudo_count,pseudo_reliable_count,pseudo_uncertain_count,pseudo_ignored_count
0,0,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,27,255,39,8,31,0
1,1,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,9,167,13,7,6,0
2,2,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,39,300,64,16,48,0
3,3,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,20,190,29,5,24,0
4,4,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,27,300,47,10,37,0
5,5,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,38,300,28,11,17,0
6,6,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,6,87,10,4,6,0
7,7,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,22,218,37,13,24,0
8,8,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,23,297,41,7,34,0
9,9,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,/app/Object_Detection/efficient_teacher/effici...,19,300,30,7,23,0


output_dir: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/visual_compare/et_default_lr1e-3


## 3. ノートブック内プレビュー

出力された比較画像をそのまま表示します。30枚だと縦に長くなります。

In [4]:
def gallery_html(paths, width=PREVIEW_WIDTH):
    rows = []
    for path in paths:
        rel = Path(path)
        rows.append(
            f'<div style="margin: 0 0 22px 0; overflow-x: auto; width: 100%;">'
            f'<div style="font-family: sans-serif; font-size: 13px; margin-bottom: 4px;">{rel.name}</div>'
            f'<img src="{rel.as_posix()}" style="width: {width}px; max-width: none; border: 1px solid #ddd;">'
            f'</div>'
        )
    return "<div>" + "\n".join(rows) + "</div>"


display(HTML(gallery_html(summary["output"].tolist())))

## 4. 別variantを見る場合

例として、pseudoGT数を制限した`et_capped_balanced_obj`を見る設定です。必要ならこのセルのコメントを外して実行してください。

In [5]:
# VARIANT = "et_capped_balanced_obj"
# cfg.variant = VARIANT
# cfg.output_dir = EXP_ROOT / "results" / "visual_compare" / VARIANT
# comparer = PseudoGTVisualComparer(cfg)
# display(comparer.describe())
# summary = comparer.run()
# display(summary)
# display(HTML(gallery_html(summary["output"].tolist())))